# Notebook 1. Weight decay 3e-6 against no weight decay, on MNIST

**About this copy.** This is the MNIST copy of `model_fitting/1_two_experiments.ipynb` on the zain branch. The methods and the order of the parts are the same. These things changed.

- The laziness knob is the initial scale `s` (column `init_scale`). It takes the place of `alpha`.
- A run has grokked when its test accuracy reaches 0.80. On modular addition the level was 0.95. The three levels used in Part 4 and Part 9 are 0.70, 0.80 and 0.85.
- Each run gets 500,000 steps.
- The two groups are weight decay 3e-6 (`M.WD_LEVEL`) and no weight decay. On modular addition they were 3e-5 and no weight decay.
- The landmark is the kernel step nearest step 5,000 (`M.LANDMARK_TARGET`). On modular addition it was step 2,430.
- Parts that compared our numbers with published modular addition numbers have no MNIST counterpart, so they are gone. Those are the comparison with report Table 2 in Part 3 and the comparison with report Table 1 in Part 8.

The data are read from `mnist/sweep/` through `mnist_settings.py`, which holds the settings that differ from modular addition. The grid values, the budget and the widths are read from the data. So the notebook runs on any part of the grid.

**The question.** Can we tell early on when a run will grok? We read three numbers about the kernel near the start of training, `A_t`, `S_t` and `R_t`. Then we check whether a simple model can use them to predict the grokking step.

**The two experiments.** We look at the runs with weight decay 3e-6 and compare them with the runs with no weight decay. Notebook 2 does the same with every weight decay level together.

**The short answer.**

**Takeaway.** To be written once the MNIST sweep has run.

### Words you need

| Word | What it means |
| --- | --- |
| step | One update of the weights. Each run gets 500,000 steps, which is the **budget**. |
| grokked | The test accuracy reached 0.80. The **grokking step** `T` is the first step where that happens, counted from step 0 (report §3.5, with the MNIST level). |
| censored | The run never reached 0.80 within the budget. We keep it, and we know only that `T` is over 500,000. |
| memorised | The training accuracy reached 0.99. |
| cell | One mix of initial scale `s`, width `N` and weight decay `ηκ`. Each cell has 5 seeds. |
| `s` | The initial scale. The initial weights are multiplied by `s`. It plays the part of `alpha` on modular addition and sets how lazy the network starts. |
| kernel | A table that says how much a training step on one input moves the output on another. We use the kernel of the summed outputs divided by the square root of 10, on a fixed draw of 1,000 test images (report Eq. 5, applied to MNIST). |
| `S_t` | How much bigger the kernel is than at step 0. 1 means the same size. |
| `R_t` | How much the kernel has turned since step 0. 0 means the same shape. |
| `A_t` | How well the kernel lines up with the labels. Higher means inputs with the same answer look more alike. |
| landmark | The kernel step nearest step 5,000. The setup cell prints it. We read the kernel there, before any run groks. |

Blue is weight decay 3e-6 and orange is no weight decay. Violet, teal and gold mark `A_t`, `S_t` and `R_t`.

**The split.** Each seed draws its own 1,000 training images, so each seed has its own split, as in report §3.1. Seed `k` uses data seed 42 + `k`. The test set is all 10,000 MNIST test images.

## Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2
from lifelines import CoxPHFitter, CoxTimeVaryingFitter, KaplanMeierFitter, LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

import mnist_settings as M

plt.style.use('fitting.mplstyle')
WD_COLOUR, NO_WD_COLOUR, GREY, INK, SURFACE = '#2a78d6', '#eb6834', '#8a8984', '#0b0b0b', '#fcfcfb'
KERNEL_COLOURS = {'A_t': '#7b5cd6', 'S_t': '#1a9e8f', 'R_t': '#c99a06'}
kernel = ['A_t', 'S_t', 'R_t']
KNOB = M.KNOB          # the initial scale s, column init_scale
print('data folder:', M.DATA)

## Load the data

We need three files. `runs.parquet` holds the settings of each run. `checkpoints.parquet` holds the accuracies at every checkpoint. `report_kernel.parquet` holds `A_t`, `S_t`, `R_t` and `D_t` at every checkpoint. `M.load()` keeps about 48 log-spaced kernel steps, as the modular addition notebooks had, and picks the landmark among them.

In [ ]:
runs, acc, kern, LANDMARK = M.load()
BUDGET = int(runs['last_step'].max())
N_SEEDS = int(runs.groupby([KNOB, 'width', 'eta_kappa']).size().max())

# The first step at which each accuracy reaches each level. NaN means it never did.
LEVEL_COLS = {f't_gen{round(100 * level)}': level for level in M.LEVELS}
T_COL = f't_gen{round(100 * M.LEVEL)}'
events = runs.set_index('run_id')
events['t_mem'] = acc[acc['train_acc'] >= M.MEM].groupby('run_id')['step'].min()
for col, level in LEVEL_COLS.items():
    events[col] = acc[acc['test_acc'] >= level].groupby('run_id')['step'].min()
events = events.reset_index()

events['grokked'] = events[T_COL].notna().astype(int)        # 1 = grokked, 0 = censored
events['T'] = events[T_COL].fillna(events['last_step'])      # the grokking step, or the budget if censored
events['cell'] = events.apply(M.cell_name, axis=1)

# Add the kernel numbers at the landmark.
data = events.merge(kern[kern['step'] == LANDMARK].drop(columns='step'), on='run_id')

WD_LABEL = 'weight decay ' + np.format_float_scientific(M.WD_LEVEL, trim='-', exp_digits=1)   # for example 3e-6
NO_WD_LABEL = 'no weight decay'
wd = data[np.isclose(data['eta_kappa'], M.WD_LEVEL, rtol=1e-6, atol=0)].copy()
no_wd = data[data['eta_kappa'] == 0].copy()
sets = {WD_LABEL: wd, NO_WD_LABEL: no_wd}
colours = {WD_LABEL: WD_COLOUR, NO_WD_LABEL: NO_WD_COLOUR}
scales = sorted(data[KNOB].unique())
widths = sorted(data['width'].unique())

# A time model (linear regression, AFT, Cox) needs grokked runs in at least two cells. A group with fewer gets none.
TIME_SETS = [label for label, df in sets.items() if df.loc[df['grokked'] == 1, 'cell'].nunique() >= 2]
print('groups with time models:', TIME_SETS or 'none')

print(f'landmark: step {LANDMARK:,}; budget: {BUDGET:,} steps; grokked means test accuracy {M.LEVEL:g}')
pd.DataFrame({label: {'runs': len(df), 'cells': df['cell'].nunique(), 'grokked': df['grokked'].sum()}
              for label, df in sets.items()})

In [ ]:
wd[['run_id', 'cell', 'T', 'grokked'] + kernel].head()

Each row is one run. `T` is the grokking step and `grokked` says whether we saw it. The last three columns are the kernel numbers at the landmark.

## Part 1. Where do the runs grok?

For each initial scale `s` and width, how many of the 5 seeds grok?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), layout='constrained')
for ax, (label, df) in zip(axes, sets.items()):
    counts = df.pivot_table(index=KNOB, columns='width', values='grokked', aggfunc='sum').reindex(index=scales, columns=widths)
    ax.imshow(counts, cmap=LinearSegmentedColormap.from_list(label, [SURFACE, colours[label]]), vmin=0, vmax=N_SEEDS,
              aspect='auto')
    for i in range(counts.shape[0]):
        for j in range(counts.shape[1]):
            n = counts.iloc[i, j]
            ax.text(j, i, '' if np.isnan(n) else f'{n:g}', ha='center', va='center',
                    color='white' if n >= 0.8 * N_SEEDS else INK)
    ax.set_xticks(range(len(widths)), widths)
    ax.set_yticks(range(len(scales)), [f'{s:g}' for s in scales])
    ax.set_xlabel('Width N')
    ax.set_title(label)
    ax.grid(False)
axes[0].set_ylabel('Initial scale s')
fig.suptitle(f'Seeds that grok, out of {N_SEEDS}', y=1.06)
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 2. How long does grokking take?

We want one typical grokking step per cell. Some seeds never grok, so a plain median would leave them out. The **Kaplan-Meier** median keeps them. A censored seed counts as "not grokked yet" for the whole budget (report Eq. 9). If fewer than half the seeds grok, the cell has no median.

In [ ]:
rows = []
for label, df in sets.items():
    for cell, g in df.groupby('cell'):
        km = KaplanMeierFitter().fit(g['T'], g['grokked'])
        rows.append({'group': label, KNOB: g[KNOB].iloc[0], 'width': g['width'].iloc[0],
                     'median T': km.median_survival_time_})
cell_medians = pd.DataFrame(rows).replace(np.inf, np.nan).sort_values(['group', KNOB, 'width'])

markers, lines = ['o', 's', '^', 'D', 'v', 'P'], ['-', '--', ':', '-.']
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, label in zip(axes, sets):
    for k, s in enumerate(scales):
        m = cell_medians[(cell_medians['group'] == label) & (cell_medians[KNOB] == s)]
        ax.plot(m['width'], m['median T'], marker=markers[k % len(markers)], linestyle=lines[k % len(lines)],
                markersize=8, color=colours[label], label=f's {s:g}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xticks(widths, widths)
    ax.minorticks_off()
    ax.set_xlabel('Width N')
    ax.set_title(label)
    ax.legend()
axes[0].set_ylabel('Median grokking step')
fig.suptitle('Median grokking step of each cell. A gap means fewer than half the seeds grok.')
plt.tight_layout()
plt.show()

The two groups may share some cells where both have a median. Here they are side by side.

In [ ]:
shared = cell_medians.pivot(index=[KNOB, 'width'], columns='group', values='median T')
shared = shared.reindex(columns=[WD_LABEL, NO_WD_LABEL]).dropna()
shared['no wd / wd'] = shared[NO_WD_LABEL] / shared[WD_LABEL]

if len(shared):
    ax = shared[[WD_LABEL, NO_WD_LABEL]].plot.bar(color=[WD_COLOUR, NO_WD_COLOUR], rot=0, figsize=(7, 3.5))
    ax.set_xticklabels([f'({s:g}, {n})' for s, n in shared.index])
    ax.set_xlabel('(s, width)')
    ax.set_ylabel('Median grokking step')
    ax.set_title('Cells where both groups have a median')
    plt.show()
else:
    print('No cell has a median in both groups.')
shared.round(2)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 3. What the kernel does during training

We follow the pilot cell (`M.PILOT`, initial scale 7 and width 200, the setup the MNIST exploration found to grok) with no weight decay, and the same cell with weight decay 3e-6. The line is the median of the 5 seeds and the band spans them, as in report Fig. 1. The dotted line is the landmark.

In [ ]:
in_pilot = (np.isclose(runs[KNOB], M.PILOT[KNOB]) & (runs['width'] == M.PILOT['width']))
pilot = {WD_LABEL: set(runs.loc[in_pilot & np.isclose(runs['eta_kappa'], M.WD_LEVEL, rtol=1e-6, atol=0), 'run_id']),
         NO_WD_LABEL: set(runs.loc[in_pilot & (runs['eta_kappa'] == 0), 'run_id'])}
assert all(pilot.values()), 'the pilot cell is missing from the data for one of the two groups'
panels = [('test_acc', 'Test accuracy', acc), ('S_t', 'Scale S_t', kern), ('R_t', 'Rotation R_t', kern),
          ('A_t', 'Alignment A_t', kern)]

fig, axes = plt.subplots(4, 2, figsize=(11, 10), sharex=True, sharey='row')
for j, (label, ids) in enumerate(pilot.items()):
    for i, (col, name, table) in enumerate(panels):
        sub = table[table['run_id'].isin(ids) & (table['step'] > 0)]
        band = sub.groupby('step')[col].agg(['min', 'median', 'max'])
        ax = axes[i, j]
        ax.fill_between(band.index, band['min'], band['max'], color=colours[label], alpha=0.2, linewidth=0)
        ax.plot(band.index, band['median'], color=colours[label], marker='.' if table is kern else None)
        ax.axvline(LANDMARK, color=GREY, linestyle=':')
        axes[i, 0].set_ylabel(name)
    axes[0, j].set_title(label)
    axes[-1, j].set_xscale('log')
    axes[-1, j].set_xlabel('Step')
axes[0, 0].axhline(M.LEVEL, color=GREY, linewidth=1)
axes[0, 1].axhline(M.LEVEL, color=GREY, linewidth=1)
fig.suptitle(f"The pilot cell (s {M.PILOT[KNOB]:g}, width {M.PILOT['width']}) with and without weight decay. "
             f"The grey line in the top row is test accuracy {M.LEVEL:g}.")
plt.tight_layout()
plt.show()

Here are the numbers of the pilot cell at step 0, at the landmark and at the last kernel step. The values are the medians over the 5 seeds. The modular addition notebook set these next to report Table 2. That table is for modular addition, and there are no published MNIST numbers, so the comparison column is gone.

In [ ]:
last_kernel_step = int(kern['step'].max())
pilot_rows = {}
for label, ids in pilot.items():
    at = kern[kern['run_id'].isin(ids) & kern['step'].isin([0, LANDMARK, last_kernel_step])]
    med = at.groupby('step')[['A_t', 'S_t', 'R_t', 'D_t']].median().T
    for step in med.columns:
        pilot_rows[(label, f'step {step:,}')] = med[step]
pd.DataFrame(pilot_rows).round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 4. No peeking

A model that predicts grokking must use only what is known before the run groks. So we read the kernel at the landmark. The check below makes sure no run, in any group, has grokked by then at any of the three levels. If it fails, lower `M.LANDMARK_TARGET` in `mnist_settings.py` so that the landmark comes earlier.

In [ ]:
earliest = events[list(LEVEL_COLS)].min()
assert (earliest.dropna() > LANDMARK).all(), (
    f'a run reached a grokking level at or before the landmark (step {LANDMARK:,}). '
    'Lower M.LANDMARK_TARGET in mnist_settings.py and run the notebook again.')
pd.Series({**{f'earliest step at test accuracy {level:g}': earliest[col] for col, level in LEVEL_COLS.items()},
           'landmark': LANDMARK,
           'highest test accuracy at the landmark': acc.loc[acc['step'] == LANDMARK, 'test_acc'].max()}).round(2)

In [ ]:
first = max(1, min(events[['t_mem'] + list(LEVEL_COLS)].min().min(), LANDMARK) / 2)   # a little before the first event
bins = np.logspace(np.log10(first), np.log10(BUDGET), 40)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, title in [(axes[0], 't_mem', f'Step where the run memorised ({M.MEM:g})'),
                       (axes[1], T_COL, f'Step where the run grokked ({M.LEVEL:g})')]:
    for label, df in sets.items():
        ax.hist(df[col].dropna(), bins=bins, color=colours[label], alpha=0.6, label=label)
    ax.axvline(LANDMARK, color=INK, linestyle='--')
    ax.text(LANDMARK, 0.95, ' landmark', transform=ax.get_xaxis_transform(), va='top')
    ax.set_xscale('log')
    ax.set_xlabel('Step')
    ax.set_title(title)
axes[0].set_ylabel('Runs')
axes[1].legend(loc='center left')
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

We count `T` from step 0. A gap counted from memorisation would already have started for runs that memorise before the landmark, and not for the others.

### How close are the seeds of one cell?

Each dot below is one seed with weight decay 3e-6, and each column is one cell.

In [ ]:
seen = wd[wd['grokked'] == 1].copy()
seen['log10 T'] = np.log10(seen['T'])
seen['name'] = 's ' + seen[KNOB].map('{:g}'.format) + ', N ' + seen['width'].astype(str)
order = seen.groupby('name')['log10 T'].mean().sort_values().index
share = seen.groupby('name')['log10 T'].transform('mean').var() / seen['log10 T'].var()

fig, ax = plt.subplots(figsize=(10, 3.8))
for i, name in enumerate(order):
    y = seen.loc[seen['name'] == name, 'log10 T']
    ax.scatter(np.full(len(y), i), y, color=WD_COLOUR, s=18)
ax.set_xticks(range(len(order)), order, rotation=90)
ax.set_ylabel('log10 grokking step')
ax.set_title(f'{WD_LABEL}: {share:.0%} of the spread is between cells')
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

Seeds of one cell share their settings, so their grokking steps can sit close together. A model tested on a seed whose cell-mates it has already seen could then be checked against a near copy of the answer. So every score below **holds out one whole cell at a time**. We fit on all the other cells and predict the held-out one.

## Part 5. The ladder of models

We try four models, from the simplest up. Each one gets three sets of inputs.

- **settings**: initial scale `s` and width, as yes/no columns.
- **kernel**: `A_t`, `S_t` and `R_t` at the landmark.
- **both**: settings and kernel together.

If the kernel tells us something the settings don't, **both** should beat **settings**.

| Step | Model | Predicts | Runs it uses | Score |
| --- | --- | --- | --- | --- |
| 1 | Linear regression | log `T` | grokked runs only | concordance |
| 2 | Logistic regression | groks within the budget, yes or no | all | AUC |
| 3 | AFT | log `T` | all, with censored runs as "over 500,000" | concordance |
| 4 | Cox | the chance of grokking at the next step | all | concordance |

**Concordance** takes every pair of runs where we know which grokked first, and counts how often the model orders them right. **AUC** is the chance that a run that grokked gets a higher score than one that didn't. For both, 0.5 is a coin toss and 1 is perfect.

A time model needs grokked runs in at least two cells, because it holds out one cell at a time. A group with fewer gets no time model, and its scores are left empty. The setup cell prints which groups qualify.

The picture below is the reason the ladder exists. Some runs may never grok, and for them we only know a lower limit for `T`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (label, df) in zip(axes, sets.items()):
    d = df.sort_values('T').reset_index(drop=True)
    seen_T = d['grokked'] == 1
    ax.scatter(d.index[seen_T], d.loc[seen_T, 'T'], color=colours[label], s=14, label='grokked: we know T')
    ax.scatter(d.index[~seen_T], d.loc[~seen_T, 'T'], color=GREY, marker='^', s=30,
               label=f'never grokked: T is over {BUDGET:,}')
    ax.set_yscale('log')
    ax.set_xlabel('Runs, sorted by T')
    ax.set_title(label)
    ax.legend(loc='lower right')
axes[0].set_ylabel('Grokking step T')
plt.tight_layout()
plt.show()

## Part 6. Step 1: linear regression

The simplest model is a straight line through log `T`. It needs a real `T`, so it has to **drop every run that never grokked**. Keep that weakness in mind.

The loop holds out one cell at a time. The kernel numbers are standardised with the training runs only. `scores` collects every held-out score in this notebook.

In [ ]:
INPUTS = ['settings', 'kernel', 'both']
scores = {}


def settings_columns(df):
    # The settings as yes/no columns, one per initial scale and width except the first.
    return pd.get_dummies(df[[KNOB, 'width']].astype('category'), drop_first=True, dtype=float)


for label, df in sets.items():
    if label not in TIME_SETS:
        print(f'{label}: grokked runs in fewer than two cells, so no linear regression.')
        for inputs in INPUTS:
            scores[(label, 'linear regression', inputs)] = np.nan
        continue
    g = df[df['grokked'] == 1].copy()
    dummies = settings_columns(g)
    for cell in g['cell'].unique():
        train, test = g['cell'] != cell, g['cell'] == cell
        z = (g[kernel] - g.loc[train, kernel].mean()) / g.loc[train, kernel].std()
        for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
            line = LinearRegression().fit(X[train], np.log10(g.loc[train, 'T']))
            g.loc[test, f'lin_{inputs}'] = line.predict(X[test])
    for inputs in INPUTS:
        scores[(label, 'linear regression', inputs)] = concordance_index(g['T'], g[f'lin_{inputs}'])

lin = pd.DataFrame({label: [scores[(label, 'linear regression', i)] for i in INPUTS] for label in sets}, index=INPUTS)
ax = lin.plot.bar(color=[colours[l] for l in lin.columns], rot=0, ylim=(0.4, 1), figsize=(7, 3.5))
ax.axhline(0.5, color=GREY, linestyle=':')
ax.set_ylabel('Held-out concordance')
ax.set_title('Linear regression, grokked runs only')
plt.show()
lin.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 7. Step 2: logistic regression

Next we keep every run but ask a simpler question: **will it grok within the budget, yes or no?** Logistic regression gives each run a probability of grokking.

This model needs runs that grok and runs that don't in the training cells of every fold. When a held-out cell holds every failed run, the model has no failed runs left to learn from. The cell below checks this for each group and fits the model only where it holds.

In [ ]:
def logistic_ok(df):
    # True when every hold-one-cell-out training set has both grokked and censored runs.
    return all(df.loc[df['cell'] != cell, 'grokked'].nunique() == 2 for cell in df['cell'].unique())


LOGIT_SETS = [label for label, df in sets.items() if logistic_ok(df)]
print('groups with a logistic model:', LOGIT_SETS or 'none')

rows = {}
for label in LOGIT_SETS:
    df = sets[label]
    dummies = settings_columns(df)
    for cell in df['cell'].unique():
        train, test = df['cell'] != cell, df['cell'] == cell
        z = (df[kernel] - df.loc[train, kernel].mean()) / df.loc[train, kernel].std()
        for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
            clf = LogisticRegression(max_iter=1000).fit(X[train], df.loc[train, 'grokked'])
            df.loc[test, f'prob_{inputs}'] = clf.predict_proba(X[test])[:, 1]
    for inputs in INPUTS:
        scores[(label, 'logistic regression', inputs)] = roc_auc_score(df['grokked'], df[f'prob_{inputs}'])
        rows[(label, inputs)] = {'AUC': scores[(label, 'logistic regression', inputs)],
                                 'balanced accuracy': balanced_accuracy_score(df['grokked'], df[f'prob_{inputs}'] > 0.5)}
for label in sets:
    if label not in LOGIT_SETS:
        for inputs in INPUTS:
            scores[(label, 'logistic regression', inputs)] = np.nan
pd.DataFrame(rows).T.round(3)

**Balanced accuracy** averages the hit rate on grokked runs and on the others, so always guessing "no" scores 0.5. A **confusion matrix** counts the right and wrong guesses, with a cut at probability 0.5.

In [ ]:
if LOGIT_SETS:
    fig, axes = plt.subplots(len(LOGIT_SETS), 3, figsize=(12, 3.4 * len(LOGIT_SETS)), squeeze=False)
    for row, label in enumerate(LOGIT_SETS):
        df = sets[label]
        for ax, inputs in zip(axes[row], INPUTS):
            guess = (df[f'prob_{inputs}'] > 0.5).astype(int)
            matrix = pd.crosstab(df['grokked'], guess).reindex(index=[0, 1], columns=[0, 1], fill_value=0).to_numpy()
            ax.imshow(matrix, cmap=LinearSegmentedColormap.from_list('n', [SURFACE, colours[label]]), vmin=0, vmax=len(df))
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, matrix[i, j], ha='center', va='center', fontsize=12)
            ax.set_xticks([0, 1], ['predicted no', 'predicted yes'])
            ax.set_yticks([0, 1], ['did not grok', 'grokked'])
            ax.set_title(f'{label}, {inputs}')
            ax.grid(False)
    fig.suptitle('Held-out guesses of the logistic model')
    plt.tight_layout()
    plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 8. Step 3: the AFT model

Linear regression drops the runs that never grok. Logistic regression keeps them but forgets the timing. The **accelerated failure time (AFT)** model does both.

This is the model in report Eq. 10 and 11, in §3.5. Eq. 10 says how log `T` depends on the inputs. Eq. 11 says how each run counts in the fit.

Think of it as a linear regression on log `T` with one upgrade. If a run grokked, we know `T`. The model fits it like normal. If it didn't, we only know `T` is over 500,000. The model uses exactly that. Each input stretches or squeezes the time. A coefficient `a` multiplies the median grokking step by e^a. So a negative `a` means sooner.

**How this differs from the report.** The model is the same. What we feed it is different.

- The report gives it the settings only. Here we add width and the kernel numbers.
- The report fits it with no penalty. We add a tiny one (0.01). Without it, the fit blows up when a setting has no grokked run.
- Our budget is 500,000 steps, and the knob is the initial scale `s`.

At the end of this part we also run the report's exact version. That way you can see both.

In [ ]:
for label, df in sets.items():
    if label not in TIME_SETS:
        print(f'{label}: grokked runs in fewer than two cells, so no AFT model.')
        for inputs in INPUTS:
            scores[(label, 'AFT', inputs)] = np.nan
        continue
    dummies = settings_columns(df)
    for cell in df['cell'].unique():
        train, test = df['cell'] != cell, df['cell'] == cell
        z = (df[kernel] - df.loc[train, kernel].mean()) / df.loc[train, kernel].std()
        for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
            aft = LogNormalAFTFitter(penalizer=0.01).fit(X[train].assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
            df.loc[test, f'aft_{inputs}'] = aft.predict_median(X[test]).to_numpy().ravel()
    for inputs in INPUTS:
        scores[(label, 'AFT', inputs)] = concordance_index(df['T'], df[f'aft_{inputs}'], df['grokked'])

aft_scores = pd.DataFrame({label: [scores[(label, 'AFT', i)] for i in INPUTS] for label in sets}, index=INPUTS)
ax = aft_scores.plot.bar(color=[colours[l] for l in aft_scores.columns], rot=0, ylim=(0.4, 1), figsize=(7, 3.5))
ax.axhline(0.5, color=GREY, linestyle=':')
ax.set_ylabel('Held-out concordance')
ax.set_title('AFT, all runs')
plt.show()
aft_scores.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### Which kernel numbers matter in the AFT?

Now we fit the **both** model on every run and read the three kernel coefficients. The **likelihood ratio test** asks whether the three numbers together improve on the settings alone. A small p-value says yes.

In [ ]:
coef_tables, lr_p = {}, {}
for label in TIME_SETS:
    df = sets[label]
    dummies = settings_columns(df)
    z = (df[kernel] - df[kernel].mean()) / df[kernel].std()
    with_kernel = LogNormalAFTFitter(penalizer=0.01).fit(dummies.join(z).assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
    without = LogNormalAFTFitter(penalizer=0.01).fit(dummies.assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
    lr_p[label] = chi2.sf(2 * (with_kernel.log_likelihood_ - without.log_likelihood_), 3)
    coef_tables[label] = with_kernel.summary.loc['mu_'].loc[kernel, ['coef', 'coef lower 95%', 'coef upper 95%', 'p']]
coefs = pd.concat(coef_tables) if coef_tables else pd.DataFrame(columns=['coef', 'coef lower 95%', 'coef upper 95%', 'p'])
coefs['times the median'] = np.exp(coefs['coef'])

fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, label in zip(axes, sets):
    if label not in coef_tables:
        ax.set_title(label)
        ax.text(0.5, 0.5, 'no time model for this group', transform=ax.transAxes, ha='center', color=GREY)
        ax.set_xticks([])
        continue
    c = coefs.loc[label]
    ax.errorbar(c['coef'], range(3), xerr=[c['coef'] - c['coef lower 95%'], c['coef upper 95%'] - c['coef']],
                fmt='o', markersize=8, color=colours[label])
    ax.axvline(0, color=GREY)
    ax.set_yticks(range(3), kernel)
    ax.set_xlabel('AFT coefficient (left of 0 = groks sooner)')
    ax.set_title(f'{label}, joint p = {lr_p[label]:.2g}')
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()
coefs.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

These intervals count each seed as an independent run, so they are likely a bit too narrow.

### The report's exact version

This is Eq. 10 the way the report fits it (§3.5). No weight decay. The only inputs are yes/no columns for the knob, here the initial scale `s`. No penalty. No kernel numbers.

The report's model has no width column. So we stick to one width, the pilot width 200. Each event gets its own fit, as the report says. That's memorised at 0.99, then grokked at 0.70, 0.80 and 0.85.

The table shows how many times longer the median step is than at the smallest `s`. That's e^a, with its 95% interval. Sometimes no run at an `s` reaches the event. Then the fit pushes `a` off to infinity, and the table says "never". The modular addition notebook compared one of these medians with report Table 1. There is no published MNIST number to compare with, so that comparison is gone.

In [ ]:
base = no_wd[no_wd['width'] == M.PILOT['width']]
base_scales = sorted(base[KNOB].unique())
knob_cols = pd.get_dummies(base[KNOB].astype('category'), prefix='s', drop_first=True, dtype=float)
names = {'t_mem': f'memorised ({M.MEM:g})', **{col: f'grokked ({level:g})' for col, level in LEVEL_COLS.items()}}
print(f'{len(base)} runs at width {M.PILOT["width"]} without weight decay')

rows = {}
for event, name in names.items():
    reached = base[event].notna().astype(int)
    row = {'runs that reach it': reached.sum()}
    if reached[base[KNOB] == base_scales[0]].sum() == 0:
        row[f'median step at s {base_scales[0]:g}'] = 'never'
        rows[name] = row
        continue
    with warnings.catch_warnings():    # the "never" fits warn that a coefficient has no finite value
        warnings.simplefilter('ignore')
        fit = LogNormalAFTFitter().fit(knob_cols.assign(T=base[event].fillna(base['last_step']), reached=reached), 'T', 'reached')
    s = fit.summary.loc['mu_']
    row[f'median step at s {base_scales[0]:g}'] = round(np.exp(s.loc['Intercept', 'coef']))
    for a in base_scales[1:]:
        if reached[base[KNOB] == a].sum() == 0:
            row[f's {a:g}'] = 'never'
        else:
            c = np.exp(s.loc[f's_{a}', ['coef', 'coef lower 95%', 'coef upper 95%']])
            row[f's {a:g}'] = f'{c.iloc[0]:.1f} times ({c.iloc[1]:.1f} to {c.iloc[2]:.1f})'
    rows[name] = row
pd.DataFrame(rows).T

**Takeaway.** To be written once the MNIST sweep has run.

## Part 9. Step 4: the Cox model

The Cox model doesn't predict a time. At every step it looks at the runs that haven't grokked yet and asks which one is most likely to grok next. That chance is the **hazard**. Each input multiplies the hazard by e^γ. So a **positive** γ means sooner, which is the opposite sign to the AFT (report Eq. 13 and 14).

We use it three ways.

- **(a)** With the kernel at the landmark, scored on held-out cells like the other models.
- **(b)** The report's own version, where the kernel changes along the run.
- **(c)** An extra that isn't in the report. It compares only the seeds of one cell, so the settings drop out.

### (a) The kernel at the landmark

In [ ]:
for label, df in sets.items():
    if label not in TIME_SETS:
        print(f'{label}: grokked runs in fewer than two cells, so no Cox model.')
        for inputs in INPUTS:
            scores[(label, 'Cox', inputs)] = np.nan
        continue
    dummies = settings_columns(df)
    for cell in df['cell'].unique():
        train, test = df['cell'] != cell, df['cell'] == cell
        z = (df[kernel] - df.loc[train, kernel].mean()) / df.loc[train, kernel].std()
        for inputs, X in [('settings', dummies), ('kernel', z), ('both', dummies.join(z))]:
            cox = CoxPHFitter(penalizer=0.01).fit(X[train].assign(T=df['T'], grokked=df['grokked']), 'T', 'grokked')
            df.loc[test, f'cox_{inputs}'] = cox.predict_partial_hazard(X[test]).to_numpy().ravel()
    for inputs in INPUTS:
        # a higher hazard means sooner, so we flip its sign before scoring
        scores[(label, 'Cox', inputs)] = concordance_index(df['T'], -df[f'cox_{inputs}'], df['grokked'])

pd.DataFrame({label: [scores[(label, 'Cox', i)] for i in INPUTS] for label in sets}, index=INPUTS).round(3)

**Takeaway.** To be written once the MNIST sweep has run.

### (b) The report's model: the kernel at every step

Here the kernel can change along the run (report Eq. 13). We cut each run into stretches between the saved kernel steps. Each stretch carries the kernel from its start, so the model never sees the future. The report calls this left-continuous. A stretch gets `event` = 1 if the run groks inside it. We start at the landmark and fit each of the MNIST grokking levels 0.70, 0.80 and 0.85.

The report's model has no penalty. It has no finite estimate when one number splits the runs that grok from the rest. A failed fit is recorded in the table and is not hidden. The modular addition notebook had a dated note here on a change of train/test splits in that sweep. It does not apply to MNIST, so it is gone.

In [ ]:
long = kern.merge(data[['run_id', 'last_step'] + list(LEVEL_COLS)], on='run_id').sort_values(['run_id', 'step'])
long['start'] = long['step']
long['next'] = long.groupby('run_id')['step'].shift(-1)     # the next saved kernel step of the same run
long = long.dropna(subset=['next'])

tv_rows, tv_fits, tv_pen_rows = {}, {}, {}
shown = None     # the stretches of the first fit, shown below
for label, df in sets.items():
    for level in LEVEL_COLS:
        lf = long[long['run_id'].isin(df['run_id'])].copy()
        lf['grokked'] = lf[level].notna().astype(int)
        if lf['grokked'].sum() == 0:
            tv_rows[(label, level)] = tv_pen_rows[(label, level)] = {'fit': 'no run reaches this level'}
            continue
        lf['T'] = lf[level].fillna(lf['last_step'])
        lf = lf[(lf['start'] >= LANDMARK) & (lf['start'] < lf['T'])]      # from the landmark until the run groks
        lf['stop'] = np.minimum(lf['next'], lf['T'])
        lf['event'] = ((lf['stop'] == lf['T']) & (lf['grokked'] == 1)).astype(int)
        assert lf['event'].sum() == lf.groupby('run_id')['grokked'].first().sum()   # one event per grokked run
        at_lm = lf[lf['start'] == LANDMARK]
        lf[kernel] = (lf[kernel] - at_lm[kernel].mean()) / at_lm[kernel].std()
        cols = dict(id_col='run_id', event_col='event', start_col='start', stop_col='stop')
        stretches = lf[['run_id', 'start', 'stop', 'event'] + kernel]
        if shown is None:
            shown = stretches
        try:
            ctv = CoxTimeVaryingFitter().fit(stretches, **cols)
            tv_fits[(label, level)] = ctv.summary
            tv_rows[(label, level)] = {**ctv.summary['coef'].to_dict(), **ctv.summary['p'].add_prefix('p ').to_dict(),
                                       'joint p': ctv.log_likelihood_ratio_test().p_value, 'fit': 'ok'}
        except Exception as error:
            tv_rows[(label, level)] = {'fit': f'no finite estimate ({type(error).__name__})'}
        # The check: the same model with a small ridge penalty, which always has a finite estimate.
        try:
            pen = CoxTimeVaryingFitter(penalizer=0.01).fit(stretches, **cols)
            tv_pen_rows[(label, level)] = {**pen.summary['coef'].to_dict(), **pen.summary['p'].add_prefix('p ').to_dict()}
        except Exception as error:
            tv_pen_rows[(label, level)] = {'fit': f'failed ({type(error).__name__})'}

shown.head() if shown is not None else 'No group has a run that reaches any of the levels.'

Above are the first stretches of one run. Below are the fits.

In [ ]:
shades = dict(zip(LEVEL_COLS, ['#c4c3bd', GREY, None]))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, label in zip(axes, sets):
    for k, (level, value) in enumerate(LEVEL_COLS.items()):
        if (label, level) not in tv_fits:    # no finite estimate, see the table
            continue
        s = tv_fits[(label, level)]
        ax.errorbar(s['coef'], np.arange(3) + 0.2 * (k - 1),
                    xerr=[s['coef'] - s['coef lower 95%'], s['coef upper 95%'] - s['coef']],
                    fmt='o', color=shades[level] or colours[label], label=f'test accuracy {value:g}')
    ax.axvline(0, color=GREY)
    ax.set_yticks(range(3), kernel)
    ax.set_xlabel('Cox coefficient (right of 0 = groks sooner)')
    ax.set_title(label)
    if any((label, level) in tv_fits for level in LEVEL_COLS):
        ax.legend(loc='lower right', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'no finite estimate', transform=ax.transAxes, ha='center', color=GREY)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()
pd.DataFrame(tv_rows).T.round(3)

**The check with a penalty.** The report's model has no penalty. Below is the same model with a small ridge penalty of 0.01, which always gives a finite answer. It is a check and is not in the report. Where the report's model has no finite estimate, this table shows which way the numbers point.

In [ ]:
pd.DataFrame(tv_pen_rows).T.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

Be careful with this model. It reads the kernel up to the step before grokking, so it shows the kernel moving while the run groks. That describes the run and gives no early warning. The landmark models are the early warnings. The report adds (§3.5) that the run itself produces these numbers, so the model says nothing about cause.

### (c) Extra: inside one cell

This Cox model gets one **stratum** per cell. Each cell has its own baseline, and the model only compares seeds of the same cell. So the settings drop out completely, and we see whether a seed with a higher kernel number groks sooner than its cell-mates.

In [ ]:
rows = {}
for label in TIME_SETS:
    df = sets[label]
    z = (df[kernel] - df[kernel].mean()) / df[kernel].std()
    cox = CoxPHFitter().fit(df[['T', 'grokked', 'cell']].join(z), 'T', 'grokked', strata='cell', robust=True)
    rows[label] = {**cox.summary['coef'].add_prefix('coef ').to_dict(), **cox.summary['p'].add_prefix('p ').to_dict(),
                   'joint p': cox.log_likelihood_ratio_test().p_value}
pd.DataFrame(rows).T.round(3)

A picture of the same idea, for the runs that grokked. Each dot is one seed. Its position shows how far its `A_t` and its grokking step sit above or below the average of its cell-mates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (label, df) in zip(axes, sets.items()):
    g = df[df['grokked'] == 1].copy()
    g = g[g.groupby('cell')['T'].transform('size') >= 2]
    ax.axhline(0, color=GREY, linewidth=1)
    ax.axvline(0, color=GREY, linewidth=1)
    ax.set_xlabel('A_t above its cell-mates (standard deviations)')
    if len(g) < 3:
        ax.set_title(f'{label}, {len(g)} runs')
        ax.text(0.5, 0.5, 'too few grokked seeds', transform=ax.transAxes, ha='center', color=GREY)
        continue
    z = (g['A_t'] - df['A_t'].mean()) / df['A_t'].std()
    x = z - z.groupby(g['cell']).transform('mean')
    y = np.log10(g['T']) - np.log10(g['T']).groupby(g['cell']).transform('mean')
    slope, intercept = np.polyfit(x, y, 1)
    ax.scatter(x, y, color=colours[label], s=16, alpha=0.7)
    ax.plot(np.sort(x), intercept + slope * np.sort(x), color=INK)
    ax.set_title(f'{label}, r = {np.corrcoef(x, y)[0, 1]:.2f}, {len(g)} runs')
axes[0].set_ylabel('log10 T above its cell-mates')
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 10. The four models side by side

First, every held-out score from Parts 6 to 9 in one table and one picture. Logistic regression is scored by AUC and the others by concordance. For both, 0.5 is a coin toss. A missing bar means the model was not fitted to that group.

In [ ]:
models = ['linear regression', 'logistic regression', 'AFT', 'Cox']
table = pd.Series(scores).unstack()[INPUTS]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, label in zip(axes, sets):
    t = table.loc[label].reindex(models)
    t.plot.bar(ax=ax, rot=0, color=['#c4c3bd', GREY, colours[label]], ylim=(0.4, 1), legend=False)
    ax.axhline(0.5, color=GREY, linestyle=':')
    ax.set_title(label)
axes[0].set_ylabel('Held-out score')
axes[1].legend(axes[1].containers, INPUTS, loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()
table.round(3)

### Which of `A_t`, `S_t` and `R_t` predicts?

Now each number goes in on its own, in every model. **Alone** shows everything the number knows, including what it knows about the settings. **On top of the settings** shows what it adds beyond them. The loop is the same hold-out loop as above. It fits many models and can take a few minutes.

In [ ]:
one = {}
for label, df in sets.items():
    dummies = settings_columns(df)
    grok = df['grokked'] == 1
    lin_ok = label in TIME_SETS      # the rule of Part 5
    for number in kernel:
        for how in ['alone', 'on top of the settings']:
            pred = pd.DataFrame(index=df.index, columns=models, dtype=float)
            for cell in df['cell'].unique():
                train, test = df['cell'] != cell, df['cell'] == cell
                z = (df[[number]] - df.loc[train, [number]].mean()) / df.loc[train, [number]].std()
                X = z if how == 'alone' else dummies.join(z)
                fit_on = X[train].assign(T=df['T'], grokked=df['grokked'])
                if lin_ok and (test & grok).any():
                    line = LinearRegression().fit(X[train & grok], np.log10(df.loc[train & grok, 'T']))
                    pred.loc[test & grok, 'linear regression'] = line.predict(X[test & grok])
                if label in LOGIT_SETS:    # see Part 7 for which groups get a logistic model
                    clf = LogisticRegression(max_iter=1000).fit(X[train], df.loc[train, 'grokked'])
                    pred.loc[test, 'logistic regression'] = clf.predict_proba(X[test])[:, 1]
                if lin_ok:
                    aft = LogNormalAFTFitter(penalizer=0.01).fit(fit_on, 'T', 'grokked')
                    pred.loc[test, 'AFT'] = aft.predict_median(X[test]).to_numpy().ravel()
                    cox = CoxPHFitter(penalizer=0.01).fit(fit_on, 'T', 'grokked')
                    pred.loc[test, 'Cox'] = -cox.predict_partial_hazard(X[test]).to_numpy().ravel()
            one[(label, how, number)] = {
                'linear regression': concordance_index(df.loc[grok, 'T'], pred.loc[grok, 'linear regression']) if lin_ok else np.nan,
                'logistic regression': roc_auc_score(df['grokked'], pred['logistic regression']) if label in LOGIT_SETS else np.nan,
                'AFT': concordance_index(df['T'], pred['AFT'], df['grokked']) if lin_ok else np.nan,
                'Cox': concordance_index(df['T'], pred['Cox'], df['grokked']) if lin_ok else np.nan}
one = pd.DataFrame(one).T.sort_index()
one.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for i, label in enumerate(sets):
    for j, how in enumerate(['alone', 'on top of the settings']):
        m = one.loc[(label, how)].T.reindex(index=models, columns=kernel)
        ax = axes[i, j]
        ax.imshow(m.astype(float), cmap=LinearSegmentedColormap.from_list('s', [SURFACE, colours[label]]), vmin=0.5, vmax=1, aspect='auto')
        for r in range(m.shape[0]):
            for c in range(m.shape[1]):
                v = m.iloc[r, c]
                ax.text(c, r, '' if np.isnan(v) else f'{v:.2f}', ha='center', va='center',
                        color='white' if v > 0.85 else INK)
        ax.set_xticks(range(3), kernel)
        ax.set_yticks(range(len(models)), models)
        ax.set_title(f'{label}, {how}')
        ax.grid(False)
fig.suptitle('Held-out score of one kernel number at a time (0.5 = coin toss)')
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 11. Do we need more runs without weight decay?

Below are, for no weight decay, the grokked seeds of each cell and the median test accuracy of each cell at the end of the budget. Cells that end close to 0.80 might grok with a longer budget. Cells with few grokked runs are thin data for any model.

In [ ]:
final = {f'Seeds that grok (out of {N_SEEDS})': no_wd.pivot_table(index='width', columns=KNOB, values='grokked', aggfunc='sum'),
         f'Median test accuracy at step {BUDGET:,}': no_wd.pivot_table(index='width', columns=KNOB, values='test_acc_final', aggfunc='median')}
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
for ax, (title, m), top in zip(axes, final.items(), [N_SEEDS, 1]):
    ax.imshow(m, cmap=LinearSegmentedColormap.from_list('n', [SURFACE, NO_WD_COLOUR]), vmin=0, vmax=top, aspect='auto')
    for r in range(m.shape[0]):
        for c in range(m.shape[1]):
            v = m.iloc[r, c]
            ax.text(c, r, f'{v:g}' if top == N_SEEDS else f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.8 * top else INK)
    ax.set_xticks(range(m.shape[1]), [f'{s:g}' for s in m.columns])
    ax.set_yticks(range(m.shape[0]), m.index)
    ax.set_xlabel('Initial scale s')
    ax.set_title(title)
    ax.grid(False)
axes[0].set_ylabel('Width N')
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Summary

**Takeaway.** To be written once the MNIST sweep has run.

Notebook 2 pools every weight decay level and checks whether what the kernel adds holds there too.